# Speech-Based Cognitive Fatigue Detection for Hiligaynon Speakers
### Model Training Pipeline: Mel-Spectrogram Dual Conv2D + Temporal Attention

**Hardware Target:** Google Colab (T4 GPU)  
**Objective:** Execute speaker-independent 5-fold cross-validation and export `fatigue_cnn_attention.keras` for web deployment.

In [ ]:
# Cell 1: Environment Verification & GPU Detection
!pip install -q librosa soundfile pydub scikit-learn

import os
import io
import numpy as np
import pandas as pd
import librosa
import soundfile as sf
import tensorflow as tf
from tensorflow.keras import layers, regularizers, Model
from sklearn.model_selection import GroupKFold
from sklearn.metrics import classification_report, confusion_matrix

# Verify GPU
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    print(f"GPU Detected: {gpus[0]}")
else:
    print("No GPU detected. In Colab, switch runtime: Runtime -> Change runtime type -> T4 GPU.")

# Set seeds for reproducibility
np.random.seed(42)
tf.random.set_seed(42)

In [ ]:
# Cell 2: Audio Preprocessing & Spectrogram Pipeline
TARGET_SR = 16000
TARGET_DURATION = 15.0
TARGET_SAMPLES = int(TARGET_SR * TARGET_DURATION)  # 240,000 samples
N_FFT = 400          # 25ms window
HOP_LENGTH = 160     # 10ms hop
N_MELS = 128
TARGET_FRAMES = 1500 # Deterministic temporal frame count

def extract_features_from_audio(file_path, top_db=20):
    """Standardize duration, perform VAD trimming, and extract 128x1500 Mel-spec."""
    y, sr = sf.read(file_path, dtype="float32")
    if y.ndim > 1:
        y = np.mean(y, axis=1)
    if sr != TARGET_SR:
        y = librosa.resample(y, orig_sr=sr, target_sr=TARGET_SR)
        
    # VAD silence trimming
    y_trimmed, _ = librosa.effects.trim(y, top_db=top_db)
    if len(y_trimmed) == 0:
        y_trimmed = y
    y_norm = librosa.util.normalize(y_trimmed)
    
    # Fixed 15.0s enforcement (pad/center-clip)
    if len(y_norm) < TARGET_SAMPLES:
        total_pad = TARGET_SAMPLES - len(y_norm)
        y_fixed = np.pad(y_norm, (total_pad // 2, total_pad - total_pad // 2), mode='constant')
    else:
        start = (len(y_norm) - TARGET_SAMPLES) // 2
        y_fixed = y_norm[start : start + TARGET_SAMPLES]
        
    # Mel-spectrogram in dB
    mel_spec = librosa.feature.melspectrogram(
        y=y_fixed, sr=TARGET_SR, n_fft=N_FFT, hop_length=HOP_LENGTH, n_mels=N_MELS, center=True
    )
    mel_spec_db = librosa.power_to_db(mel_spec, ref=np.max)
    
    # Slice or pad temporal frames to exactly 1500
    if mel_spec_db.shape[1] >= TARGET_FRAMES:
        mel_spec_db = mel_spec_db[:, :TARGET_FRAMES]
    else:
        pad_w = TARGET_FRAMES - mel_spec_db.shape[1]
        mel_spec_db = np.pad(mel_spec_db, ((0, 0), (0, pad_w)), mode='constant', constant_values=mel_spec_db.min())
        
    return mel_spec_db.astype(np.float32)

In [ ]:
# Cell 3: Neural Network Architecture & Serializable Temporal Attention
try:
    from tensorflow.keras.utils import register_keras_serializable
except ImportError:
    import keras
    register_keras_serializable = keras.saving.register_keras_serializable

@register_keras_serializable(package="CustomLayers")
class TemporalAttention(layers.Layer):
    def __init__(self, units=64, **kwargs):
        super().__init__(**kwargs)
        self.units = units

    def build(self, input_shape):
        feature_dim = input_shape[-1]
        self.w = self.add_weight(shape=(feature_dim, self.units), initializer="glorot_uniform", trainable=True)
        self.b = self.add_weight(shape=(self.units,), initializer="zeros", trainable=True)
        self.v = self.add_weight(shape=(self.units, 1), initializer="glorot_uniform", trainable=True)
        super().build(input_shape)

    def call(self, inputs):
        score = tf.nn.tanh(tf.matmul(inputs, self.w) + self.b)
        energy = tf.matmul(score, self.v)
        weights = tf.nn.softmax(energy, axis=1)
        context = tf.reduce_sum(inputs * weights, axis=1)
        return context, tf.squeeze(weights, axis=-1)

    def get_config(self):
        cfg = super().get_config()
        cfg.update({"units": self.units})
        return cfg

def build_fatigue_model(input_shape=(128, 1500, 1), num_classes=3, l2_reg=1e-4):
    inputs = layers.Input(shape=input_shape, name="mel_input")
    
    # Conv Block 1
    x = layers.Conv2D(32, (3, 3), padding="same", kernel_regularizer=regularizers.l2(l2_reg))(inputs)
    x = layers.BatchNormalization()(x)
    x = layers.Activation("relu")(x)
    x = layers.MaxPooling2D((2, 2))(x)
    x = layers.Dropout(0.2)(x)
    
    # Conv Block 2
    x = layers.Conv2D(64, (3, 3), padding="same", kernel_regularizer=regularizers.l2(l2_reg))(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation("relu")(x)
    x = layers.MaxPooling2D((2, 2))(x)
    x = layers.Dropout(0.2)(x)
    
    # Reshape for Temporal Attention
    x = layers.Permute((2, 1, 3))(x)
    time_steps = x.shape[1]
    feat_dim = x.shape[2] * x.shape[3]
    x = layers.Reshape((time_steps, feat_dim))(x)
    
    context, _ = TemporalAttention(units=64, name="temporal_attention")(x)
    
    # Classification Head
    d = layers.Dense(64, activation="relu", kernel_regularizer=regularizers.l2(l2_reg))(context)
    d = layers.Dropout(0.3)(d)
    outputs = layers.Dense(num_classes, activation="softmax", name="fatigue_output")(d)
    
    return Model(inputs=inputs, outputs=outputs, name="fatigue_cnn_attention")

In [ ]:
# Cell 4: Synthetic/Supabase Data Generator & Label Mapping
def map_samn_perelli_label(score):
    """1-2: Low (0), 3-4: Moderate (1), 5-7: High (2)"""
    if score <= 2:
        return 0
    elif score <= 4:
        return 1
    return 2

# Simulating 90 records across 30 participants for notebook pipeline verification
print("Generating dataset tensors for 30 respondents (90 total sessions)...")
num_samples = 90
X = np.random.randn(num_samples, 128, 1500, 1).astype(np.float32)
groups = np.repeat([f"WVSU_CS_{i:03d}" for i in range(1, 31)], 3)
y_raw_scores = np.random.randint(1, 8, size=num_samples)
y_classes = np.array([map_samn_perelli_label(s) for s in y_raw_scores])
y_cat = tf.keras.utils.to_categorical(y_classes, num_classes=3)

print(f"X shape: {X.shape}, y shape: {y_cat.shape}, Unique speakers: {len(np.unique(groups))}")

In [ ]:
# Cell 5: Speaker-Independent GroupKFold Cross-Validation
gkf = GroupKFold(n_splits=5)
fold_accuracies = []

for fold, (train_idx, val_idx) in enumerate(gkf.split(X, y_cat, groups=groups), 1):
    print(f"\n--- Training Fold {fold}/5 ---")
    X_train, y_train = X[train_idx], y_cat[train_idx]
    X_val, y_val = X[val_idx], y_cat[val_idx]
    
    model = build_fatigue_model()
    model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
                  loss="categorical_crossentropy",
                  metrics=["accuracy"])
    
    early_stop = tf.keras.callbacks.EarlyStopping(monitor="val_loss", patience=5, restore_best_weights=True)
    
    history = model.fit(X_train, y_train,
                        validation_data=(X_val, y_val),
                        epochs=50,
                        batch_size=16,
                        callbacks=[early_stop],
                        verbose=0)
    
    val_loss, val_acc = model.evaluate(X_val, y_val, verbose=0)
    fold_accuracies.append(val_acc)
    print(f"Fold {fold} - Val Loss: {val_loss:.4f} | Val Accuracy: {val_acc * 100:.2f}%")

print(f"\nMean 5-Fold Validation Accuracy: {np.mean(fold_accuracies) * 100:.2f}% (+/- {np.std(fold_accuracies) * 100:.2f}%)")

In [ ]:
# Cell 6: Holdout Speaker Evaluation & Model Artifact Export
# Train final production model on full dataset
final_model = build_fatigue_model()
final_model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
                    loss="categorical_crossentropy",
                    metrics=["accuracy"])

final_model.fit(X, y_cat, epochs=15, batch_size=16, verbose=1)

# Export deployment model
export_path = "fatigue_cnn_attention.keras"
final_model.save(export_path)
print(f"Successfully exported deployment model to {export_path}")

# Verify reloadability
loaded_model = tf.keras.models.load_model(export_path)
test_preds = loaded_model.predict(X[:3])
print("Reload verification predictions shape:", test_preds.shape)